# 4. Research Supervisor Multi-Agent Coordination

This notebook demonstrates multi-agent coordination using a Research Supervisor that decomposes briefs into parallel research assignments executed concurrently.

In [ ]:
import os
import asyncio
from typing import List, Dict, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

## 1. Supervisor Delegation Schemas

In [ ]:
class ConductResearch(BaseModel):
    topic: str = Field(description="Specific sub-topic for worker agent.")
    search_queries: List[str] = Field(description="2 to 3 targeted search queries.")
    depth: str = Field(default="in-depth", description="Target depth.")

class SupervisorPlan(BaseModel):
    tasks: List[ConductResearch] = Field(description="Parallel assignments to delegate.")
    rationale: str = Field(description="Coordination rationale.")

class ResearchNote(BaseModel):
    topic: str
    summary: str
    key_findings: List[str] = Field(default_factory=list)
    sources: List[Dict[str, str]] = Field(default_factory=list)

class SupervisorState(BaseModel):
    topic: str
    sub_topics: List[str]
    delegated_tasks: List[ConductResearch] = Field(default_factory=list)
    completed_notes: List[ResearchNote] = Field(default_factory=list)

## 2. Supervisor Planning & Parallel Worker Execution

In [ ]:
def get_llm():
    return ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        temperature=0.2,
    )

async def plan_delegation_node(state: SupervisorState) -> Dict[str, Any]:
    llm = get_llm().with_structured_output(SupervisorPlan)
    prompt = f"Research Topic: {state.topic}\nSub-Topics: {state.sub_topics}\nCreate parallel research tasks."
    plan = await llm.ainvoke([
        SystemMessage(content="You are a Research Supervisor delegating sub-topics to parallel worker agents."),
        HumanMessage(content=prompt)
    ])
    return {"delegated_tasks": plan.tasks}

async def execute_subagent_task(task: ConductResearch) -> ResearchNote:
    """Worker function executed concurrently with context isolation."""
    llm = get_llm()
    prompt = f"Analyze the sub-topic: '{task.topic}' using queries: {task.search_queries}. Provide 3 key findings."
    resp = await llm.ainvoke([
        SystemMessage(content="You are an autonomous sub-topic research specialist."),
        HumanMessage(content=prompt)
    ])
    return ResearchNote(
        topic=task.topic,
        summary=resp.content,
        key_findings=[l.strip("- ") for l in resp.content.split("\n") if l.strip().startswith("-")],
        sources=[{"title": f"{task.topic} Analysis", "url": "https://example.com/source"}]
    )

async def parallel_execution_node(state: SupervisorState) -> Dict[str, Any]:
    results = await asyncio.gather(
        *[execute_subagent_task(t) for t in state.delegated_tasks]
    )
    return {"completed_notes": list(results)}

## 3. Build and Test Supervisor Graph

In [ ]:
builder = StateGraph(SupervisorState)
builder.add_node("plan", plan_delegation_node)
builder.add_node("execute_parallel", parallel_execution_node)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute_parallel")
builder.add_edge("execute_parallel", END)
supervisor_graph = builder.compile()

initial_state = SupervisorState(
    topic="High-Temperature Superconductivity Mechanisms and Applications",
    sub_topics=["Cuprates vs Iron-Based Superconductors", "Room-Temperature Hydrides under Pressure", "Practical Industrial Power Grid & Maglev Applications"]
)

result = await supervisor_graph.ainvoke(initial_state)
print(f"Completed {len(result['completed_notes'])} parallel sub-agent research tracks:")
for note in result["completed_notes"]:
    print(f"\n--- {note.topic} ---")
    print(note.summary[:200] + "...")